In [1]:
# STEP 1: Install
!pip install pandas numpy scikit-learn xgboost

In [2]:
# STEP 2: Imports
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb

In [3]:
# STEP 3: Load dataset
data = fetch_openml(name='adult', version=2, as_frame=True)
df = data.frame

df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
# STEP 4: Convert target to binary (CTR-style)
df['target'] = df['class'].apply(lambda x: 1 if x == '>50K' else 0)

In [5]:
# Drop original column
df = df.drop('class', axis=1)

In [6]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

In [7]:
# STEP 5: Encode categorical features
X = pd.get_dummies(X, drop_first=True)

In [8]:
# STEP 6: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
# STEP 7: Model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [10]:
# STEP 8: Evaluation
preds = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, preds)

print("ROC-AUC:", auc)

ROC-AUC: 0.9321978687340565


In [11]:
# STEP 9: Simulated CTR + Bid Optimization
df_test = X_test.copy()
df_test['click_prob'] = preds

In [12]:
# Simulated bidding strategy
df_test['bid_value'] = df_test['click_prob'] * 100

df_test[['click_prob', 'bid_value']].head()

,click_prob,bid_value
7762,0.011807,1.180685
23881,0.068515,6.851477
30507,0.995679,99.567863
28911,0.187704,18.770428
19484,0.652226,65.222565
